In [3]:
%pip -q install "qiskit==2.2.3" "qiskit-algorithms==0.4.0" "qiskit-optimization==0.7.0" "koreanize-matplotlib==0.1.1"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [11]:
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import Statevector
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

# 1. 입력 데이터
materials = [
    "철판A", "나사B", "베어링C", "코팅제D", "포장재E",
    "전선F", "윤활유G", "필터H", "접착제I", "플라스틱J"
]
unit_cost = np.array([120, 15, 60, 90, 5, 25, 80, 35, 40, 30])
daily_demand = np.array([35, 120, 25, 15, 180, 50, 10, 18, 20, 60])
current_stock = np.array([40, 300, 10, 5, 600, 20, 8, 15, 5, 100])
lead_time = np.array([3, 2, 5, 4, 1, 3, 6, 4, 5, 2])
importance = np.array([5, 3, 4, 5, 2, 4, 5, 4, 3, 2])
delivery_risk = np.array([80, 10, 60, 75, 5, 55, 90, 50, 70, 20])

# 2. 업무 지표 계산
shortage = np.maximum(daily_demand*lead_time-current_stock, 0)
risk = shortage*(importance/5)*(delivery_risk/100)
cost_n = unit_cost/unit_cost.max()
risk_n = risk/max(float(risk.max()), 1.0)
alpha, beta = 0.3, 0.7

# 3. 페널티 항을 포함한 목적함수 구성 (슬랙 비트 제거로 속도 극대화)
problem = QuadraticProgram()
for name in materials:
    problem.binary_var(name)

# 기본 비용 함수: 발주 비용 + 위험 회피
# 부족량이 0인 품목은 발주하면 큰 페널티 부여
penalty = 10.0
linear_coef = alpha * cost_n - beta * risk_n
for i, name in enumerate(materials):
    if shortage[i] == 0:
        linear_coef[i] += penalty  # 재고 충분한 품목 발주 차단 페널티

problem.minimize(constant=float(beta * risk_n.sum()), linear=linear_coef)
# 최대 4개 발주 제약은 페널티(또는 사후 필터링)로 처리하여 슬랙 큐비트 폭증 방지

# 4. QUBO와 Ising 해밀토니안 (큐비트 수 10개 유지)
converter = QuadraticProgramToQubo()
qubo = converter.convert(problem)
hamiltonian, offset = qubo.to_ising()
num_qubits = qubo.get_num_vars()
print("최적화 큐비트 수:", num_qubits)

# 5. 표준 QAOA 회로 (Layers=1 유지)
layers = 1
gamma = ParameterVector("gamma", layers)
beta = ParameterVector("beta", layers)
circuit = QuantumCircuit(num_qubits)
circuit.h(range(num_qubits))
for layer in range(layers):
    circuit.append(PauliEvolutionGate(hamiltonian, time=gamma[layer]), range(num_qubits))
    circuit.rx(2 * beta[layer], range(num_qubits))

# 6. 기대 에너지 계산과 각도 최적화 (maxiter 단축)
def state_at(angles):
    bindings = dict(zip([*gamma, *beta], angles))
    return Statevector.from_instruction(circuit.assign_parameters(bindings))

def expected_energy(angles):
    return float(state_at(angles).expectation_value(hamiltonian).real)

initial_angles = np.concatenate((np.full(layers, 0.5), np.full(layers, 0.8)))
optimization = minimize(expected_energy, initial_angles, method="COBYLA",
                        options={"maxiter": 30})  # 반복 횟수를 줄여서 속도 향상
print("최적화 완료 상태:", optimization.message)

# 7. 최종 상태 측정
shots = 1024  # 샷 수도 절반으로 축소
final_state = state_at(optimization.x)
final_state.seed(42)
counts = final_state.sample_counts(shots)

# 8. 결과 복원 및 검증 (무거운 전수조사 제거)
feasible_samples = []
for bitstring, count in counts.items():
    qubo_bits = np.array([int(bit) for bit in reversed(bitstring)])
    bits = np.asarray(converter.interpret(qubo_bits), dtype=int)
    # 사후 제약 검증 (최대 4개 이하 & 부족량 있는 것만)
    if bits.sum() <= 4 and not np.any((bits == 1) & (shortage == 0)):
        value = float(problem.objective.evaluate(bits))
        feasible_samples.append((value, tuple(bits), count))

if not feasible_samples:
    # 만약 조건에 맞는 샘플이 없으면 큐비트 상태 중 가장 비용이 낮은 것 선택
    bitstring = max(counts, key=counts.get)
    qubo_bits = np.array([int(bit) for bit in reversed(bitstring)])
    chosen = np.asarray(converter.interpret(qubo_bits), dtype=int)
else:
    _, chosen_tuple, _ = min(feasible_samples)
    chosen = np.array(chosen_tuple)

print("선택 비트:", chosen)
print("발주 품목:", [name for name, b in zip(materials, chosen) if b > 0.5])
print("발주 종류 수:", chosen.sum())

# 9. 결과 저장
np.savez('inventory_result.npz', names=np.array(materials), chosen=chosen,
         shortage=shortage, risk=risk)
print("결과 저장 완료 (`inventory_result.npz`)")

최적화 큐비트 수: 10
최적화 완료 상태: Return from COBYLA because the objective function has been evaluated MAXFUN times.
선택 비트: [0 0 1 0 0 1 1 0 1 0]
발주 품목: ['베어링C', '전선F', '윤활유G', '접착제I']
발주 종류 수: 4
결과 저장 완료 (`inventory_result.npz`)
